In [2]:
# imports
import pickle
import os
import re 
import mne
import numpy as np
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import sklearn.metrics as metrics
import scipy as sp
import scipy.stats as stats
import warnings
from sklearn.pipeline import make_pipeline
from toeplitzlda.classification import ToeplitzLDA
import logging
import math
import sys
import line_profiler

# so we can access the modules in utils/
current = os.path.dirname(os.path.realpath("utils"))
print(current)
parent = os.path.dirname(current)
print(parent)
sys.path.append(parent)


# utils functions
from utils.preprocessing import all_have_same_condition, load_complete_session, inspect_session, get_n_epochs, get_iteration_structure, load_session_chached
from utils.feature_extraction import get_jumping_means, epoch_vectorizer_channelprime
from utils.offline_evaluation import compare_auc_single_trial_interval, compute_auc_with_cv
from utils.online_simulation import online_simulation

# Turn off warnings (that most likely occur from ToeplitzLDA)
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
mne.set_log_level('WARNING')



C:\Users\Soz\OneDrive - Radboud Universiteit\Bureaublad\BSc_Thesis\Code\BSc_Thesis_Project\ERP_analysis_code\patient_pipeline\notes
C:\Users\Soz\OneDrive - Radboud Universiteit\Bureaublad\BSc_Thesis\Code\BSc_Thesis_Project\ERP_analysis_code\patient_pipeline


### pp

## ⚠️ important: when loading sessions

Copy paste the module preprocessing.py and replace all data_dir by:

```
data_dir = Path.cwd().parent / data_path
```
(instead of `data_dir = Path.cwd() / data_path`)

### Line profiler

In [ ]:
%load_ext line_profiler

def prof_function(x):
    my_list = []
    for i in range(x):
        if (i>5):
            my_list.append(i*i)
        else:
            my_list.append(i) 

%lprun -f prof_function prof_function(500000)
#%lprun -f load_session_chached load_session_chached("data_p1/P1_S5/anonymized")
#%lprun -f compare_auc_single_trial_interval compare_auc_single_trial_interval(trials_s5)


In [4]:
# data_s5 = load_session_chached("data_p1/P1_S5/anonymized")

# print(data_s5.keys())
# trials_s5 = data_s5.get('trials')
# #print(len(trials_s5))
# #compare_auc_single_trial_interval(trials_s5)
# print(data_s5.get('timestamp'))
# print(data_s5.get('filenames'))

# # trials = trials_s5
# # calibration_trials = trials[0:12]
# # online_trials = trials[12:]
# # %lprun -f online_simulation online_simulation(calibration_trials, online_trials, log_process="testtesttest.log")

## Online simulation 

#### Some information about the variables:
- `online_trial_targets` contains the target word id ($[1,2, ..,6]$) per trial. These should **only** be used to quantify the performance.
- `online_labels` contains whether the presented stimulus/word is a target (1) or a non-target (0). Note that the order of stimuli differs per iteration.
- `online_words` contains the word id ($[1,2, ..,6]$) per stimulus/word presentation. 

1. Inspect the code given below. In this code the classifier predicts the signed distance to the decision boundary, given a single presented stimulus/word. You will build on top of this code, so make sure you understand the three variables listed above.

1. **Condition A: no dynamic stopping applied.** Predict (using the provided classifier `clf`) the target words per trial, using the entire trial information. In other words, keep track of the **signed distances** to the decision hyperplane per word id throughout the entire trial to make a single prediction about the target word after all 15 iterations. Report your prediction accuracy. 
**NOTE**: The order in which words are presented differs per iteration in a trial. Also note that the classifier tried to learn a decision hyperplane such that targets have a positive signed distance to the hyperplane, and non-targets have a negative signed distance to the hyperplane.